# rawi V3 — two-stage gated model (presence + value heads)

An experiment that **internalises the agreement-gating** that makes the
`text2tashkeel` ensemble strong (docs §9) into a *single* model. Instead of one
73-way head, a shared BiLSTM encoder feeds **two heads**:

1. **presence head** (binary: does this letter take a mark?) — trained on **all**
   real positions. This is the *gate* (the *where*).
2. **value head** (which mark) — trained **only on marked positions**. This is the
   *value* (the *which*) — exactly what rawi is already best at (DER* ≈ 3%).

**Inference:** the presence head decides where to mark; where it says "mark",
the value head supplies it; elsewhere, nothing. One model, no external gate.

Changes vs V1/V2 are in the **model**, **training**, and **inference** cells
(marked 🔱); data loading and the tokenizer are unchanged. Targets are padded
with `-100` as in V2.


# Arabic Diacritization Model
This notebook trains a small model to predict Arabic diacritics, exports to ONNX, and benchmarks performance.

## 1. Configuration - Environment Variables
Set all hyperparameters and configuration via environment variables or defaults

In [ ]:
import os

# ============================================================================
# FILE PATHS
# ============================================================================
TRAIN_FILE = os.getenv('TRAIN_FILE', '/datasets/arabic-diacritics/train.txt')
VAL_FILE = os.getenv('VAL_FILE', '/datasets/arabic-diacritics/val.txt')
TEST_FILE = os.getenv('TEST_FILE', '/datasets/arabic-diacritics/test.txt')


# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================
# Options: 'lstm', 'gru', 'transformer'
MODEL_TYPE = os.getenv('MODEL_TYPE', 'lstm')

# Output files
MODEL_PATH = os.getenv('MODEL_PATH', f'diacritization_model_{MODEL_TYPE}.pth')
ONNX_PATH = os.getenv('ONNX_PATH', f'diacritization_model_{MODEL_TYPE}.onnx')

# Embedding & Hidden Dimensions
EMBEDDING_DIM = int(os.getenv('EMBEDDING_DIM', '128'))
HIDDEN_DIM = int(os.getenv('HIDDEN_DIM', '256'))
NUM_LAYERS = int(os.getenv('NUM_LAYERS', '2'))

# Transformer-specific parameters
NUM_HEADS = int(os.getenv('NUM_HEADS', '4'))  # For transformer
FF_DIM = int(os.getenv('FF_DIM', '256'))  # Feedforward dim for transformer

# Regularization
DROPOUT = float(os.getenv('DROPOUT', '0.3'))

# ============================================================================
# TRAINING HYPERPARAMETERS
# ============================================================================
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '512'))
EPOCHS = int(os.getenv('EPOCHS', '50'))
LEARNING_RATE = float(os.getenv('LEARNING_RATE', '0.001'))
WEIGHT_DECAY = float(os.getenv('WEIGHT_DECAY', '1e-5'))
MAX_SEQ_LENGTH = int(os.getenv('MAX_SEQ_LENGTH', '500'))

# Learning rate scheduler
USE_LR_SCHEDULER = os.getenv('USE_LR_SCHEDULER', 'true').lower() == 'true'
LR_SCHEDULER_PATIENCE = int(os.getenv('LR_SCHEDULER_PATIENCE', '2'))
LR_SCHEDULER_FACTOR = float(os.getenv('LR_SCHEDULER_FACTOR', '0.5'))

# Early stopping
USE_EARLY_STOPPING = os.getenv('USE_EARLY_STOPPING', 'true').lower() == 'true'
EARLY_STOPPING_PATIENCE = int(os.getenv('EARLY_STOPPING_PATIENCE', '2'))

# Gradient clipping
GRAD_CLIP = float(os.getenv('GRAD_CLIP', '5.0'))

# ============================================================================
# DATA LOADING
# ============================================================================
NUM_WORKERS = int(os.getenv('NUM_WORKERS', '2'))
USE_MEMORY_EFFICIENT_LOADING = os.getenv('USE_MEMORY_EFFICIENT_LOADING', 'true').lower() == 'true'

# ============================================================================
# BENCHMARKING
# ============================================================================
BENCHMARK_ITERATIONS = int(os.getenv('BENCHMARK_ITERATIONS', '100'))

# ============================================================================
# DEVICE
# ============================================================================
DEVICE = os.getenv('DEVICE', 'auto')  # 'auto', 'cpu', or 'cuda'

# ============================================================================
# PRINT CONFIGURATION
# ============================================================================
print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
print(f"\nFiles:")
print(f"  Train: {TRAIN_FILE}")
print(f"  Val: {VAL_FILE}")
print(f"  Test: {TEST_FILE}")
print(f"  Model output: {MODEL_PATH}")
print(f"  ONNX output: {ONNX_PATH}")

print(f"\nModel Architecture:")
print(f"  Type: {MODEL_TYPE.upper()}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Hidden dim: {HIDDEN_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
if MODEL_TYPE == 'transformer':
    print(f"  Num heads: {NUM_HEADS}")
    print(f"  Feedforward dim: {FF_DIM}")
print(f"  Dropout: {DROPOUT}")

print(f"\nTraining:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Gradient clipping: {GRAD_CLIP}")
print(f"  LR scheduler: {USE_LR_SCHEDULER}")
print(f"  Early stopping: {USE_EARLY_STOPPING}")

print(f"\nData Loading:")
print(f"  Memory-efficient: {USE_MEMORY_EFFICIENT_LOADING}")
print(f"  Num workers: {NUM_WORKERS}")

print("\n" + "=" * 70)

## 2. Install Dependencies

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu --break-system-packages
!pip install onnx onnxruntime pyarabic --break-system-packages

## 3. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import re
import time
import unicodedata
from collections import Counter
import onnx
import onnxruntime as ort
from pathlib import Path
import math
from tqdm import tqdm

# Set device
if DEVICE == 'auto':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
else:
    device = torch.device(DEVICE)
    
print(f'Using device: {device}')

## 4. Arabic Diacritics Utilities with Unicode Normalization

In [ ]:
# Arabic diacritics (combining marks in Unicode)
ARABIC_DIACRITICS = {
    '\u0064': 'FATHA',           # َ Fatha
    '\u064B': 'TANWIN_FATH',    # ً Tanwin Fath
    '\u064F': 'DAMMA',           # ُ Damma
    '\u064C': 'TANWIN_DAMM',    # ٌ Tanwin Damm
    '\u0650': 'KASRA',           # ِ Kasra
    '\u064D': 'TANWIN_KASR',    # ٍ Tanwin Kasr
    '\u0652': 'SUKUN',           # ْ Sukun
    '\u0651': 'SHADDA',          # ّ Shadda
}

DIACRITIC_CHARS = ''.join(ARABIC_DIACRITICS.keys())


def normalize_text(text):
    """Normalize text using NFD and strip emojis across all languages."""
    # Step 1: Normalize to NFD (separates base characters from diacritics)
    normalized = unicodedata.normalize('NFD', text)
    
    # Step 2: Filter out characters categorized as Symbols (So)
    # This captures emojis while preserving letters, numbers, and punctuation.
    return "".join(c for c in normalized if unicodedata.category(c) != 'So')


def remove_diacritics(text):
    """Remove all Arabic diacritics from normalized text"""
    # Normalize first to ensure diacritics are separate
    text = normalize_text(text)
    # Remove combining marks (category Mn = Mark, nonspacing)
    return ''.join(char for char in text if unicodedata.category(char) != 'Mn')


def extract_diacritics(text):
    """Extract diacritics aligned with characters using Unicode normalization"""
    # Normalize to NFD so diacritics are separate combining characters
    text = normalize_text(text)
    result = []
    i = 0
    
    while i < len(text):
        char = text[i]
        
        # Skip if this is already a combining mark
        if unicodedata.category(char) == 'Mn':
            i += 1
            continue
        
        # Collect all following combining marks (diacritics)
        diacritics = []
        j = i + 1
        while j < len(text) and unicodedata.category(text[j]) == 'Mn':
            diacritics.append(text[j])
            j += 1
        
        # Store character and its diacritics
        result.append((char, ''.join(diacritics)))
        i = j
    
    return result

# Test the normalization
test_text = " القَلْعَ مَرْحَبًا"
#test_text = "وَإِنْ وَهَبَهَا لِرَبِّ الأَرْضِ لَمْ يَلْزَمْهُ القَبُولُ إنْ أَرَادَ القَلْعَ ،"
print(f"Original: {test_text}")
print(f"Normalized (NFD): {normalize_text(test_text)}")
print(f"Without diacritics: {remove_diacritics(test_text)}")
print(f"Extracted pairs: {extract_diacritics(test_text)}")

## 5. Memory-Efficient Data Loading

In [ ]:
def count_lines(file_path):
    """Count lines in a file without loading it all into memory"""
    count = 0
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                count += 1
    return count

# Count sentences in each file
print("Counting sentences in files...")
train_count = count_lines(TRAIN_FILE)
val_count = count_lines(VAL_FILE)
test_count = count_lines(TEST_FILE)

print(f"Training sentences: {train_count}")
print(f"Validation sentences: {val_count}")
print(f"Test sentences: {test_count}")
print(f"Total: {train_count + val_count + test_count}")

# For vocabulary building, we need to scan all files once
# But we'll do it efficiently without storing all pairs
print("\nBuilding vocabulary (streaming through files)...")

## 6. Build Vocabulary (Streaming)

In [ ]:
import json
import os
from tqdm import tqdm  # Optional: for progress bar

def build_vocabulary_from_file(file_paths, vocab_file='vocab.json'):
    """
    Build vocabulary with caching.
    If vocab_file exists, load from it. Otherwise, build, save, and return.
    """
    
    # 1. Load from cache if available
    if os.path.exists(vocab_file):
        print(f"Loading vocabulary from {vocab_file}...")
        try:
            with open(vocab_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                return data['char_to_idx'], data['diac_to_idx']
        except Exception as e:
            print(f"Error loading cache: {e}. Rebuilding...")

    # 2. Build from scratch
    print("Building vocabulary (streaming through files)...")
    all_chars = set()
    all_diacritics = set()
    
    for file_path in file_paths:
        print(f"Processing {file_path}...")
        with open(file_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                # OPTIMIZATION: extract_diacritics already handles normalization 
                # and separates base chars. We don't need separate calls 
                # to normalize_text() or remove_diacritics().
                char_diac_pairs = extract_diacritics(line)
                
                # Update sets directly from the pairs (1 pass instead of 3)
                for char, diacs in char_diac_pairs:
                    all_chars.add(char)
                    all_diacritics.add(diacs)
    
    # 3. Create Mappings
    # Build character vocabulary
    char_to_idx = {'<PAD>': 0, '<UNK>': 1}
    for char in sorted(all_chars):
        char_to_idx[char] = len(char_to_idx)
    
    # Build diacritic vocabulary
    diac_to_idx = {'': 0}  # No diacritic
    for diac in sorted(all_diacritics):
        if diac:  # Skip empty string as it's already added
            diac_to_idx[diac] = len(diac_to_idx)
    
    # 4. Save to cache
    print(f"Saving vocabulary to {vocab_file}...")
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump({
            'char_to_idx': char_to_idx, 
            'diac_to_idx': diac_to_idx
        }, f, ensure_ascii=False, indent=2)
    
    return char_to_idx, diac_to_idx


# This will create 'vocab.json' the first time, and load it instantly the next time
char_to_idx, diac_to_idx = build_vocabulary_from_file(
    [TRAIN_FILE, VAL_FILE, TEST_FILE], 
    vocab_file='vocab.json'
)

idx_to_char = {v: k for k, v in char_to_idx.items()}
idx_to_diac = {v: k for k, v in diac_to_idx.items()}


print(f"Character vocabulary size: {len(char_to_idx)}")
print(f"Diacritic vocabulary size: {len(diac_to_idx)}")
print(f"\nSample characters: {list(char_to_idx.keys())[2:12]}")
print(f"Diacritic classes: {list(diac_to_idx.keys())[:10]}")

## 7. Memory-Efficient Dataset Class

In [ ]:
class ArabicDiacritizationDataset(Dataset):
    """Memory-efficient dataset that reads and processes data on-demand"""
    
    def __init__(self, file_path, char_to_idx, diac_to_idx, max_length):
        self.file_path = file_path
        self.char_to_idx = char_to_idx
        self.diac_to_idx = diac_to_idx
        self.max_length = max_length
        
        # Build an index of line positions for random access
        self.line_offsets = []
        with open(file_path, 'r', encoding='utf-8') as f:
            while True:
                # Capture the exact position before reading the line
                offset = f.tell()
                line = f.readline()
                
                # End of file
                if not line:
                    break
                
                # Only index non-empty lines
                if line.strip():
                    self.line_offsets.append(offset)
        
        self.length = len(self.line_offsets)
    
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        # Read specific line from file
        with open(self.file_path, 'r', encoding='utf-8') as f:
            f.seek(self.line_offsets[idx])
            line = f.readline().strip()
        
        # Normalize the text
        line = normalize_text(line)
        
        # Process on-demand
        undiacritized = remove_diacritics(line)
        char_diac_pairs = extract_diacritics(line)
        
        # Encode characters
        chars = undiacritized[:self.max_length]
        char_indices = [self.char_to_idx.get(c, self.char_to_idx['<UNK>']) for c in chars]
        
        # Encode diacritics
        diac_indices = []
        for char, diacs in char_diac_pairs[:self.max_length]:
            diac_idx = self.diac_to_idx.get(diacs, 0)
            diac_indices.append(diac_idx)
        
        # Pad sequences
        seq_len = len(char_indices)
        char_indices += [0] * (self.max_length - seq_len)
        diac_indices += [-100] * (self.max_length - seq_len)  # -100 = ignore (padding)
        
        return {
            'chars': torch.tensor(char_indices, dtype=torch.long),
            'diacritics': torch.tensor(diac_indices, dtype=torch.long),
            'length': seq_len
        }

# Create memory-efficient datasets
print("Creating datasets with on-demand loading...")
train_dataset = ArabicDiacritizationDataset(TRAIN_FILE, char_to_idx, diac_to_idx, MAX_SEQ_LENGTH)
val_dataset = ArabicDiacritizationDataset(VAL_FILE, char_to_idx, diac_to_idx, MAX_SEQ_LENGTH)
test_dataset = ArabicDiacritizationDataset(TEST_FILE, char_to_idx, diac_to_idx, MAX_SEQ_LENGTH)

# Use num_workers for faster loading (loads batches in parallel)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=2)

print(f"\nDataset sizes:")
print(f"  Training: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")
print(f"\nMemory usage: Data is loaded on-demand, not stored in RAM")

## 3. Configuration

## 8. Model Architectures
Multiple architecture options: LSTM (bidirectional), GRU (bidirectional), and Transformer

🔱 **Two-head model.** A shared bidirectional LSTM encoder with a **presence** head (1 logit, binary) and a **value** head (`diac_size` logits). Replaces V1's single head.

In [ ]:
import torch
import torch.nn as nn

class TwoStageDiacritizationModel(nn.Module):
    """Shared BiLSTM encoder -> {presence head, value head}.

    presence head: P(this letter carries ANY mark)   -- the gate (WHERE)
    value head:    P(which mark | it carries one)    -- the value (WHICH)
    """
    def __init__(self, vocab_size, diac_size, embedding_dim, hidden_dim,
                 num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True,
                            dropout=dropout if num_layers > 1 else 0)
        self.dropout = nn.Dropout(dropout)
        self.presence_head = nn.Linear(hidden_dim * 2, 1)        # binary gate
        self.value_head = nn.Linear(hidden_dim * 2, diac_size)   # which mark

    def forward(self, x):
        h, _ = self.lstm(self.embedding(x))
        h = self.dropout(h)
        presence_logit = self.presence_head(h).squeeze(-1)  # (B, T)
        value_logits = self.value_head(h)                    # (B, T, diac_size)
        return presence_logit, value_logits

model = TwoStageDiacritizationModel(
    len(char_to_idx), len(diac_to_idx), EMBEDDING_DIM, HIDDEN_DIM,
    NUM_LAYERS, DROPOUT).to(device)
print(model)
print('params:', sum(p.numel() for p in model.parameters()))


## 9. Training Loop

🔱 **Two-head training.** Two losses on the shared encoder:

* **presence** — `BCEWithLogitsLoss` over **every real position** (target = 1 if the gold letter carries a mark, else 0). Padding (`-100`) masked out.
* **value** — `CrossEntropyLoss` over **marked positions only** (target = the gold class; unmarked positions masked out).

Total loss = presence + value. This is the trained analogue of the inference-time gate(where)+value(which) split.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

presence_criterion = nn.BCEWithLogitsLoss()
value_criterion = nn.CrossEntropyLoss()  # ignore_index=-100 (unmarked/padding)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)


def split_targets(diac):
    """From padded diac ids (-100 = pad): presence target + masked value target."""
    real = diac != -100                      # real (non-pad) positions
    marked = real & (diac != 0)              # positions that carry a mark
    presence_t = marked.float()              # 1 where marked, 0 where bare
    value_t = diac.clone()
    value_t[~marked] = -100                  # value loss only on marked positions
    return real, presence_t, value_t


def run_epoch(model, loader, train):
    model.train() if train else model.eval()
    tot = 0.0
    correct = total = 0
    for batch in loader:
        chars = batch['chars'].to(device)
        diac = batch['diacritics'].to(device)
        real, presence_t, value_t = split_targets(diac)
        with torch.set_grad_enabled(train):
            presence_logit, value_logits = model(chars)
            # presence loss over real positions only
            pl = presence_criterion(presence_logit[real], presence_t[real])
            vl = value_criterion(value_logits.view(-1, value_logits.size(-1)),
                                 value_t.view(-1))
            loss = pl + vl
            if train:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
        tot += loss.item()
        # combined accuracy over ALL real positions (gate AND value)
        gate = (torch.sigmoid(presence_logit) > 0.5)
        pred = torch.where(gate, value_logits.argmax(-1),
                           torch.zeros_like(diac))  # 0 = no mark
        m = real
        correct += (pred[m] == diac[m]).sum().item(); total += m.sum().item()
    return tot / len(loader), correct / max(total, 1)


best = float('inf')
for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(model, train_loader, True)
    va_loss, va_acc = run_epoch(model, val_loader, False)
    print(f'epoch {epoch+1}: train {tr_loss:.4f}/{tr_acc:.4f}  val {va_loss:.4f}/{va_acc:.4f}')
    if va_loss < best:
        best = va_loss; torch.save(model.state_dict(), MODEL_PATH)
        print('  saved')


🔱 **Two-stage inference.** presence head gates; value head fills the approved positions. Marks are applied to letters only (whitespace/punctuation preserved).

In [ ]:
import unicodedata as _ud

def is_arabic_letter(ch):
    return _ud.category(ch) in ('Lo', 'Ll', 'Lu', 'Lt', 'Lm')

def predict_two_stage(text, model, char_to_idx, idx_to_diac, device,
                      max_length=MAX_SEQ_LENGTH, threshold=0.5):
    model.eval()
    text = normalize_text(text); text = remove_diacritics(text)
    result = []
    for start in range(0, len(text), max_length):
        chunk = text[start:start + max_length]
        ids = [char_to_idx.get(c, char_to_idx['<UNK>']) for c in chunk]
        ids += [0] * (max_length - len(ids))
        with torch.no_grad():
            pres, val = model(torch.tensor([ids], device=device))
            gate = (torch.sigmoid(pres)[0] > threshold).cpu().numpy()
            cls = val.argmax(-1)[0].cpu().numpy()
        for i, ch in enumerate(chunk):
            result.append(ch)
            if is_arabic_letter(ch) and gate[i]:
                result.append(idx_to_diac.get(int(cls[i]), ''))
    return ''.join(result)

# example
for s in ['بسم الله الرحمن الرحيم', 'العلم نور والجهل ظلام']:
    print(predict_two_stage(remove_diacritics(s), model, char_to_idx, idx_to_diac, device))


## 10. Export to ONNX

In [ ]:
# Prepare dummy input for ONNX export
dummy_input = torch.randint(0, len(char_to_idx), (1, MAX_SEQ_LENGTH)).to(device)

# Export (works the same for all model types - they all have the same input/output interface)
model.eval()
print(f"Exporting {MODEL_TYPE.upper()} model to ONNX...")

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size', 1: 'sequence'},
        'output': {0: 'batch_size', 1: 'sequence'}
    }
)

print(f"Model exported to {ONNX_PATH}")

# Verify ONNX model
onnx_model = onnx.load(ONNX_PATH)
onnx.checker.check_model(onnx_model)
print("ONNX model verified successfully!")
print(f"\nONNX model works identically for all architectures (LSTM/GRU/Transformer)")
print(f"Input shape: (batch_size, sequence_length)")
print(f"Output shape: (batch_size, sequence_length, {len(diac_to_idx)})")

## 11. Benchmark (PyTorch vs ONNX)

In [ ]:
def benchmark_pytorch(model, device, num_iterations=BENCHMARK_ITERATIONS):
    """Benchmark PyTorch model"""
    model.eval()
    test_input = torch.randint(0, len(char_to_idx), (1, MAX_SEQ_LENGTH)).to(device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(10):
            _ = model(test_input)
    
    # Benchmark
    times = []
    with torch.no_grad():
        for _ in range(num_iterations):
            start = time.time()
            _ = model(test_input)
            times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

def benchmark_onnx(onnx_path, num_iterations=BENCHMARK_ITERATIONS):
    """Benchmark ONNX model"""
    session = ort.InferenceSession(onnx_path)
    test_input = np.random.randint(0, len(char_to_idx), (1, MAX_SEQ_LENGTH), dtype=np.int64)
    
    # Warmup
    for _ in range(10):
        _ = session.run(None, {'input': test_input})
    
    # Benchmark
    times = []
    for _ in range(num_iterations):
        start = time.time()
        _ = session.run(None, {'input': test_input})
        times.append(time.time() - start)
    
    return np.mean(times), np.std(times)

# Run benchmarks
print(f"\nBenchmarking {MODEL_TYPE.upper()} model ({BENCHMARK_ITERATIONS} iterations)...")
print("="*60)

print("\nPyTorch model...")
pytorch_mean, pytorch_std = benchmark_pytorch(model, device)
print(f"  Mean: {pytorch_mean*1000:.2f}ms")
print(f"  Std: {pytorch_std*1000:.2f}ms")

print("\nONNX model...")
onnx_mean, onnx_std = benchmark_onnx(ONNX_PATH)
print(f"  Mean: {onnx_mean*1000:.2f}ms")
print(f"  Std: {onnx_std*1000:.2f}ms")

speedup = pytorch_mean / onnx_mean
print(f"\nSpeedup: {speedup:.2f}x")

# Throughput calculation
pytorch_throughput = 1.0 / pytorch_mean
onnx_throughput = 1.0 / onnx_mean

print(f"\nThroughput:")
print(f"  PyTorch: {pytorch_throughput:.2f} sequences/second")
print(f"  ONNX: {onnx_throughput:.2f} sequences/second")
print("\n" + "="*60)

## 12. Test Inference

In [ ]:
import unicodedata as _ud


def is_arabic_letter(ch):
    """Return True only for actual letters that can carry a diacritic.

    Spaces, digits, and punctuation are explicitly excluded so they
    never receive a diacritic prediction, which preserves whitespace
    in the output string.
    """
    return _ud.category(ch) in ('Lo', 'Ll', 'Lu', 'Lt', 'Lm')


def predict_diacritics(text, model, char_to_idx, idx_to_diac, device,
                       max_length=MAX_SEQ_LENGTH):
    """Predict diacritics for undiacritised Arabic text.

    Fixes vs previous version:
    * Spaces and punctuation never receive a diacritic  -> whitespace preserved.
    * Texts longer than max_length are chunked           -> no IndexError.
    """
    model.eval()

    # Normalise and strip any existing diacritics
    text = normalize_text(text)
    text = remove_diacritics(text)

    result = []

    # Process in non-overlapping chunks so we never exceed max_length
    for chunk_start in range(0, len(text), max_length):
        chunk = text[chunk_start:chunk_start + max_length]

        char_indices = [char_to_idx.get(c, char_to_idx['<UNK>']) for c in chunk]
        padded = char_indices + [0] * (max_length - len(char_indices))

        with torch.no_grad():
            input_tensor = torch.tensor([padded], dtype=torch.long).to(device)
            output = model(input_tensor)
            # squeeze(0) gives shape (max_length,) regardless of seq len
            predictions = output.argmax(dim=-1).squeeze(0).cpu().numpy()

        for i, char in enumerate(chunk):
            result.append(char)
            # only attach diacritics to actual letters
            if is_arabic_letter(char):
                diac = idx_to_diac.get(int(predictions[i]), '')
                if diac:
                    result.append(diac)

    return ''.join(result)


# Test on test samples (read from file on-demand)
print("Testing predictions on test samples:\n")
test_samples = []
with open(TEST_FILE, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        line = line.strip()
        if line:
            line = normalize_text(line)
            test_samples.append(line)

for i, original in enumerate(test_samples):
    undiacritized = remove_diacritics(original)
    predicted = predict_diacritics(undiacritized, model, char_to_idx, idx_to_diac, device)

    print(f"Sample {i+1}:")
    print(f"Undiacritized: {undiacritized}")
    print(f"Original:      {original}")
    print(f"Predicted:     {predicted}")
    print()


## 13. Test Set Evaluation

In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")
test_loss, test_acc = evaluate(model, test_loader, criterion, device)

print(f"Test Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}")
print(f"  Test Error Rate: {(1-test_acc)*100:.2f}%")

## 14. Model Summary

In [ ]:
import os

print("=" * 70)
print("MODEL SUMMARY")
print("=" * 70)
print(f"\nDataset:")
print(f"  Training sentences: {train_count}")
print(f"  Validation sentences: {val_count}")
print(f"  Test sentences: {test_count}")
print(f"  Total: {train_count + val_count + test_count} sentences")
print(f"  Memory-efficient loading: {USE_MEMORY_EFFICIENT_LOADING}")

print(f"\nModel Architecture: {MODEL_TYPE.upper()}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
if MODEL_TYPE.lower() in ['lstm', 'gru']:
    print(f"  Hidden dim: {HIDDEN_DIM}")
    print(f"  Bidirectional: Yes")
elif MODEL_TYPE.lower() == 'transformer':
    print(f"  Num heads: {NUM_HEADS}")
    print(f"  Feedforward dim: {FF_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"  Dropout: {DROPOUT}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

print(f"\nTraining Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Gradient clipping: {GRAD_CLIP}")
print(f"  LR scheduler: {USE_LR_SCHEDULER}")
print(f"  Early stopping: {USE_EARLY_STOPPING}")

print(f"\nVocabulary:")
print(f"  Character vocab size: {len(char_to_idx)}")
print(f"  Diacritic classes: {len(diac_to_idx)}")

print(f"\nPerformance:")
print(f"  Final validation accuracy: {val_acc:.4f}")
print(f"  Test accuracy: {test_acc:.4f}")

print(f"\nText Processing:")
print(f"  Unicode normalization: NFD (canonical decomposition)")
print(f"  Diacritic detection: Using unicodedata.category (Mn)")

print(f"\nFiles:")
print(f"  PyTorch model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1024:.2f} KB)")
print(f"  ONNX model: {ONNX_PATH} ({os.path.getsize(ONNX_PATH)/1024:.2f} KB)")

print("\n" + "=" * 70)

## 15. Visualisations


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

# Human-readable names for the most common Arabic diacritics
DIAC_NAMES = {
    '': 'None',
    '\u064e': 'Fatha (a)',
    '\u064b': 'Tanwin Fath',
    '\u064f': 'Damma (u)',
    '\u064c': 'Tanwin Damm',
    '\u0650': 'Kasra (i)',
    '\u064d': 'Tanwin Kasr',
    '\u0652': 'Sukun',
    '\u0651': 'Shadda',
}

fig = plt.figure(figsize=(18, 20))
fig.suptitle(
    f"Arabic Diacritization — {MODEL_TYPE.upper()} Model  |  "
    f"Val Acc {val_acc*100:.2f}%  |  Test Acc {test_acc*100:.2f}%",
    fontsize=14, fontweight='bold', y=0.99
)

epochs_range = list(range(1, len(train_losses) + 1))

# ── 1. Loss curves ─────────────────────────────────────────────────────────
ax1 = fig.add_subplot(3, 3, 1)
ax1.plot(epochs_range, train_losses, 'o-', color='steelblue', label='Train', lw=2)
ax1.plot(epochs_range, val_losses,   's-', color='tomato',    label='Val',   lw=2)
ax1.set_title('Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Cross-entropy loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# ── 2. Accuracy curves ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(3, 3, 2)
ax2.plot(epochs_range, [a*100 for a in train_accs], 'o-', color='steelblue', label='Train', lw=2)
ax2.plot(epochs_range, [a*100 for a in val_accs],   's-', color='tomato',    label='Val',   lw=2)
ax2.axhline(test_acc*100, color='purple', linestyle='--', lw=1.5,
            label=f'Test {test_acc*100:.2f}%')
ax2.set_title('Diacritic Accuracy per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

# ── 3. Learning-rate schedule ──────────────────────────────────────────────
ax3 = fig.add_subplot(3, 3, 3)
ax3.plot(epochs_range, learning_rates, 'D-', color='darkorange', lw=2)
ax3.set_title('Learning Rate Schedule')
ax3.set_xlabel('Epoch')
ax3.set_ylabel('LR')
ax3.set_yscale('log')
ax3.grid(True, alpha=0.3)

# ── 4. Inference latency (PyTorch vs ONNX) ─────────────────────────────────
ax4 = fig.add_subplot(3, 3, 4)
_labels = ['PyTorch', 'ONNX']
_means  = [pytorch_mean*1000, onnx_mean*1000]
_stds   = [pytorch_std*1000,  onnx_std*1000]
_colors = ['steelblue', 'seagreen']
bars4 = ax4.bar(_labels, _means, yerr=_stds, color=_colors, capsize=6, width=0.4)
for bar, val in zip(bars4, _means):
    ax4.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + max(_stds)*0.15,
             f'{val:.2f} ms', ha='center', va='bottom', fontweight='bold')
ax4.set_title('Inference Latency (mean ± std, batch=1)')
ax4.set_ylabel('ms')
ax4.grid(True, alpha=0.3, axis='y')

# ── 5. Throughput ──────────────────────────────────────────────────────────
ax5 = fig.add_subplot(3, 3, 5)
_thru = [pytorch_throughput, onnx_throughput]
bars5 = ax5.bar(_labels, _thru, color=_colors, width=0.4)
for bar, val in zip(bars5, _thru):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height()*1.02,
             f'{val:.0f}/s', ha='center', va='bottom', fontweight='bold')
ax5.set_title('Throughput (sequences / second)')
ax5.set_ylabel('seq/s')
ax5.grid(True, alpha=0.3, axis='y')

# ── 6. Diacritic-class coverage in vocabulary ──────────────────────────────
ax6 = fig.add_subplot(3, 3, 6)
_class_labels = []
for k in sorted(diac_to_idx, key=lambda x: diac_to_idx[x])[:15]:
    _class_labels.append(DIAC_NAMES.get(k, repr(k)[:10]))
_x = range(len(_class_labels))
ax6.bar(_x, [1]*len(_class_labels), color='mediumpurple', width=0.6)
ax6.set_xticks(list(_x))
ax6.set_xticklabels(_class_labels, rotation=45, ha='right', fontsize=8)
ax6.set_title(f'Registered Diacritic Classes ({len(diac_to_idx)} total)')
ax6.set_ylabel('Present in vocabulary')
ax6.grid(True, alpha=0.3, axis='y')

# ── 7. Per-sample diacritic error rate ────────────────────────────────────
ax7 = fig.add_subplot(3, 3, 7)
_cer = []
_slabels = []
for i, original in enumerate(test_samples[:5]):
    undiacritized = remove_diacritics(original)
    predicted = predict_diacritics(undiacritized, model, char_to_idx, idx_to_diac, device)
    orig_pairs = extract_diacritics(normalize_text(original))
    pred_pairs = extract_diacritics(normalize_text(predicted))
    n = min(len(orig_pairs), len(pred_pairs))
    if n > 0:
        errors = sum(1 for (_, od), (_, pd) in zip(orig_pairs[:n], pred_pairs[:n]) if od != pd)
        _cer.append(errors / n * 100)
    else:
        _cer.append(0.0)
    _slabels.append(f"S{i+1}")
bars7 = ax7.bar(_slabels, _cer, color='coral', width=0.5)
for bar, val in zip(bars7, _cer):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax7.set_title('Per-Sample Diacritic Error Rate')
ax7.set_xlabel('Test sample')
ax7.set_ylabel('Error rate (%)')
ax7.grid(True, alpha=0.3, axis='y')

# ── 8. Parameter distribution by layer group ──────────────────────────────
ax8 = fig.add_subplot(3, 3, 8)
_param_groups = {}
for name, p in model.named_parameters():
    top = name.split('.')[0]
    _param_groups[top] = _param_groups.get(top, 0) + p.numel()
ax8.pie(
    list(_param_groups.values()),
    labels=list(_param_groups.keys()),
    autopct='%1.1f%%',
    startangle=140,
    colors=['steelblue', 'tomato', 'seagreen', 'darkorange', 'mediumpurple'][:len(_param_groups)]
)
ax8.set_title(f'Parameter Distribution\n({sum(_param_groups.values()):,} total params)')

# ── 9. Val − Train loss gap (overfitting monitor) ──────────────────────────
ax9 = fig.add_subplot(3, 3, 9)
_gap = [v - t for t, v in zip(train_losses, val_losses)]
ax9.bar(epochs_range, _gap,
        color=['seagreen' if g < 0 else 'tomato' for g in _gap])
ax9.axhline(0, color='black', lw=0.8)
ax9.set_title('Val − Train Loss Gap\n(negative = val better than train)')
ax9.set_xlabel('Epoch')
ax9.set_ylabel('Loss gap')
ax9.grid(True, alpha=0.3, axis='y')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('diacritization_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plots saved to diacritization_plots.png")
